# Testing using MS-SWD & MDS

This notebook generates realizations from trained generator models, loads test samples, computes the MS-SWD distances between them, and projects those distances using Multidimensional Scaling (MDS) to construct the MS-SWD embedding.

### Imports

In [9]:
import re
import h5py
import json
from pathlib import Path
from functools import partial
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.manifold import MDS

import torch
import torch.nn as nn

from voxgan.networks import resnet
from voxgan.data.datasets import *
from voxgan.data.fluvdeposet import FluvDepoSetDataset
from voxgan.models.metrics import MSSWD

### Paths

In [10]:
model_dir_path = Path(r'C:\Users\mathi\Desktop\TU Delft\TU Delft year 5\Master Thesis\Thesis-project-DGM\outputs\post_training\20000_training_samples\RUN_10000_of_20000_samples_128xy_dataset_50_epochs_bs_32_val_size_020_onr_hot')
test_data_dir_path = Path(r'C:\Users\mathi\Desktop\TU Delft\TU Delft year 5\Master Thesis\Thesis-project-DGM\datasets\testing\testing_dataset_upper_plain_delta_128')
output_dir_path = Path(r'C:\Users\mathi\Desktop\TU Delft\TU Delft year 5\Master Thesis\Thesis-project-DGM\plots\post_training_plots\RUN_10000_of_20000_samples_128xy_dataset_50_epochs_bs_32_val_size_020_onr_hot')

# Ensure the output directory exists
output_dir_path.mkdir(parents=True, exist_ok=True)

### Settings & GPU Setup

In [11]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.set_default_device(device)

np.random.seed(42)
torch.manual_seed(42)

# Determine number of GPUs dynamically
available_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
n_gpu_for_metric = min(2, available_gpus) if available_gpus > 0 else 0
print(f"Using device: {device}")
print(f"Available GPUs: {available_gpus}, allocated for MSSWD: {n_gpu_for_metric}")

Using device: cuda:0
Available GPUs: 1, allocated for MSSWD: 1


### Checkpoint Detection

In [12]:
# Search for checkpoints
checkpoint_paths = []
for i in range(3):
    model_name = 'fluvgan_1_training_1_architecture_dcgan_4_' + str(i + 1) + '_training_checkpoint_*'
    paths = sorted(list(model_dir_path.glob(model_name)), key=lambda s: int(re.findall(r'\d+', str(s))[-1]))
    if paths:
        checkpoint_paths.append(paths[-1])

# Fallback: search for any .pt files if the default naming scheme isn't matched
if not checkpoint_paths:
    print("Default checkpoint naming pattern not matched. Searching for any .pt files...")
    checkpoint_paths = sorted(list(model_dir_path.glob("*.pt")))

n_models = len(checkpoint_paths)
if n_models == 0:
    raise FileNotFoundError(f"No checkpoint (.pt) files found in: {model_dir_path}")

print(f"Found {n_models} model(s) to evaluate:")
for idx, path in enumerate(checkpoint_paths):
    print(f"  Model {idx + 1}: {path.name}")

Default checkpoint naming pattern not matched. Searching for any .pt files...
Found 1 model(s) to evaluate:
  Model 1: architecture_4_dcgan_training_datset_upper_plane_delta_20000_one_hot_epochs_50_bs_32_run_1.pt


### Load Test Samples

In [14]:
class NpyOrH5Dataset(torch.utils.data.Dataset):
    def __init__(self, root, transform=None, mapping_config_path=None):
        self.root = Path(root)
        self.transform = transform
        
        # Support both .npy and .h5 files
        self.file_paths = sorted(list(self.root.glob("*.npy")) + list(self.root.glob("*.h5")))
        self.len = len(self.file_paths)
        
        # Load facies mapping array
        self.mapping = None
        if mapping_config_path and Path(mapping_config_path).exists():
            try:
                with open(mapping_config_path, 'r') as f:
                    config = json.load(f)
                self.mapping = np.array(config['forward_mapping_array'])
                print(f"Loaded facies mapping array of length {len(self.mapping)}.")
            except Exception as e:
                print(f"Failed to load mapping config: {e}")
        
        if self.mapping is None:
            self.mapping = np.zeros(15, dtype=np.int64)
            self.mapping[1:4] = 0
            self.mapping[4:8] = 1
            self.mapping[8:15] = 2
            
    def __len__(self):
        return self.len
        
    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        if file_path.suffix == '.npy':
            data = np.load(file_path)
        elif file_path.suffix == '.h5':
            with h5py.File(file_path, 'r') as file:
                data = file['model'][:]
        else:
            raise ValueError(f"Unsupported file format: {file_path.suffix}")
            
        # If the loaded array is 3D (Z, Y, X), it lacks a channel/one-hot dimension.
        # Convert it to a 4D one-hot encoded representation (C, Z, Y, X) to match expectations of transforms.
        if len(data.shape) == 3:
            # Map raw facies values to classes (0, 1, 2)
            mapped_data = self.mapping[data.astype(np.int64)]
            
            # One-hot encode in NumPy
            num_classes = int(self.mapping.max() + 1)
            one_hot = np.eye(num_classes)[mapped_data]      # shape: (Z, Y, X, C)
            one_hot = np.moveaxis(one_hot, -1, 0)           # shape: (C, Z, Y, X)
            
            # Scale to [-1, 1]
            data = (one_hot * 2.0) - 1.0
            
        sample = {'data': data}
        if self.transform is not None:
            sample = self.transform(sample)
        return sample

transform = [
    # Adjust these tuples so the difference always matches your intended spatial dimensions
    Compose([Crop(((1, 3), (0, 16), (0, 128), None)), FillNaN((0., 'max+1')), Scale(((0, 1), None)), ToTensor()]),
    Compose([Crop(((1, 3), (0, 16), (0, 128), None)), FillNaN((0., 'max+1')), Scale(((0, 1), None)), ToTensor()]), 
]

mapping_config_path = model_dir_path / 'facies_mapping_config.json'
dataset = NpyOrH5Dataset(test_data_dir_path, transform=transform[0], mapping_config_path=mapping_config_path)

embedding = np.full((n_models*2*len(transform)*len(dataset), 100 + 5), np.nan)
test_samples = torch.empty((len(transform)*len(dataset), 2, 16, 128, 128))

for i in tqdm(range(len(transform))):
    dataset = NpyOrH5Dataset(test_data_dir_path, transform=transform[i], mapping_config_path=mapping_config_path)
    for j, test_sample in enumerate(tqdm(dataset, leave=False)):
        test_samples[i*len(dataset) + j] = test_sample['data']
        
        # Extract the sample number robustly using regular expressions
        filename = Path(dataset.file_paths[j]).stem
        match = re.search(r'\d+', filename)
        sample_id = int(match.group()) if match else j
        
        for k in range(n_models):
            embedding[k*2*len(transform)*len(dataset) + i*len(dataset) + j, 0] = k + 1
            embedding[k*2*len(transform)*len(dataset) + i*len(dataset) + j, 1] = sample_id
            embedding[k*2*len(transform)*len(dataset) + i*len(dataset) + j, 2] = i + 1

Loaded facies mapping array of length 13.


  0%|          | 0/2 [00:00<?, ?it/s]

Loaded facies mapping array of length 13.


 50%|█████     | 1/2 [00:06<00:06,  6.10s/it]

Loaded facies mapping array of length 13.


100%|██████████| 2/2 [00:15<00:00,  7.63s/it]


### Setup Model & Sampling Parameters

In [15]:
n_samples = len(transform)*len(dataset)
batch_size = 100
distances = np.zeros((n_models, 2*n_samples, 2*n_samples))

### Generate Realizations & Compute MS-SWD & MDS

In [16]:
for i in range(n_models):
    checkpoint_path = checkpoint_paths[i]
    print(f"\n--- Processing Model {i + 1} ({checkpoint_path.name}) ---")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    generator = resnet.DeepGenerator3d(nz=100,
                                       ngf=64,
                                       nc=3,
                                       nl=(3, 5, 5),
                                       max_factor=16,
                                       residual_weight=1.,
                                       mode='nearest',
                                       kernel_size=3,
                                       layer_normalization=nn.BatchNorm3d,
                                       last_layer_normalization=nn.BatchNorm3d,
                                       weight_normalization=nn.utils.parametrizations.spectral_norm,
                                       activation=partial(nn.LeakyReLU, negative_slope=0.2, inplace=True),
                                       last_activation=nn.Tanh,
                                       use_double_conv=False,
                                       use_double_resblocks=False,
                                       use_attention=False,
                                       skip_z=False,
                                       split_z=False)
    generator = nn.DataParallel(generator)
    generator.load_state_dict(checkpoint['generator'])
    generator = generator.module
    generator.eval()
    generator.requires_grad_ = False

    # Query generator outputs to obtain exact channel count dynamically
    with torch.no_grad():
        dummy_z = torch.zeros((1, generator.nz))
        dummy_out = generator(dummy_z.view(dummy_z.shape + (1, 1, 1)))
        generator_nc = dummy_out.shape[1]
    print(f"Generator output channels: {generator_nc}")

    samples = torch.empty((n_samples, 2, 16, 128, 128))
    with torch.no_grad():
        for j in tqdm(range(0, n_samples, batch_size)):
            z = torch.randn((min(batch_size, n_samples - j), generator.nz))
            s = slice(i*2*n_samples + n_samples + j,
                      i*2*n_samples + n_samples + min(j + batch_size, n_samples))
            embedding[s, 0] = i + 1
            embedding[s, 3:generator.nz + 3] = z.cpu().numpy()
            
            # Generate realizations
            gen_out = generator(z.view(z.shape + (1, 1, 1)))
            
            # If generator outputs 3 channels, slice out channels 1 and 2 (sandy facies)
            if generator_nc == 3:
                gen_out = gen_out[:, 1:3]
                
            samples[j:min(j + batch_size, n_samples)] = gen_out

    samples = torch.cat((test_samples, samples))

    ms_swd = MSSWD(n_levels=3,
                   n_descriptors=512,
                   descriptor_size=(3, 7, 7),
                   n_repeat=12,
                   n_proj=128,
                   padding_mode='circular',
                   combine_levels=True,
                   n_gpu=n_gpu_for_metric)

    print("Computing pairwise MS-SWD distances...")
    for j in tqdm(range(samples.shape[0])):
        for k in range(j + 1, samples.shape[0]):
            distance = ms_swd(samples[j:j + 1], samples[k:k + 1])
            distances[i, j, k] = distance
            distances[i, k, j] = distance

    print("Reducing dimensions using MDS...")
    reducer = MDS(n_components=2,
                  n_jobs=8,
                  random_state=42,
                  dissimilarity='precomputed')
    embedding[i*2*n_samples:(i + 1)*2*n_samples, -2:] = reducer.fit_transform(distances[i])

    print('MDS stress:', reducer.stress_)


--- Processing Model 1 (architecture_4_dcgan_training_datset_upper_plane_delta_20000_one_hot_epochs_50_bs_32_run_1.pt) ---


C:\Users\Public\Documents\Wondershare\CreatorTemp\ipykernel_23136\258023115.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_

Generator output channels: 3


  0%|          | 0/2 [00:30<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.12 GiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 8.76 GiB is allocated by PyTorch, and 78.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Save Results

In [ ]:
columns = ['Model', 'Test_sample', 'Transform']
columns += ['Latent_' + str(i + 1) for i in range(100)]
columns += ['Dimension_1', 'Dimension_2']
embedding_df = pd.DataFrame(data=embedding, columns=columns)
embedding_df = embedding_df.convert_dtypes()

csv_out = output_dir_path / 'fluvgan_1_training_2_test-msswd_embedding.csv'
npy_out = output_dir_path / 'fluvgan_1_training_2_test-msswd_distance.npy'

embedding_df.to_csv(csv_out, index=False)
np.save(npy_out, distances)

print(f"Saved embedding table to: {csv_out}")
print(f"Saved distance matrix to: {npy_out}")